In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.2 which is incompatible.
ydata-synthetic 1.4.0 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
ydata-synthetic 1.4.0 requires tensorflow==2.15.*, but you have tensorflow 2.12.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

data = data.replace("unknown", np.nan)

data[target_col] = data[target_col].astype(str).str.strip()
data[target_col] = data[target_col].map({"yes": 1, "no": 0})

date_cols = ["day_of_week", "month", "duration"]
data = data.drop(columns=[col for col in date_cols if col in data.columns])

data = data.dropna(subset=[target_col])

n_samples = min(10000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()

if target_col in numeric_cols:
    numeric_cols.remove(target_col)

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

data[target_col] = data[target_col].astype(int)

X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

metadata.update_column(
    column_name=target_col,
    sdtype="categorical"
)

N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


In [3]:
seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)


================ SINGLE RUN ================
Training TabDDPM...
[0]
35
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(35)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.2237 Sum: 0.2237
Step 1000/1000 MLoss: 0.0 GLoss: 0.1915 Sum: 0.1915
mlp
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Sample timestep    0
Discrete cols: [4]
Num shape:  (10000, 5)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:01<00:00,  7.07it/s]|
Column Shapes Score: 18.27%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 217.45it/s]|
Column Pair Trends Score: 11.42%

Overall Score (Average): 14.85%

TabDDPM: 0.1485


In [4]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()

Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:13<00:00,  1.02it/s]|
Column Shapes Score: 67.26%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:01<00:00, 54.40it/s]|
Column Pair Trends Score: 57.71%

Overall Score (Average): 62.48%

ForestDiffusion: 0.6248


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            stratify_train = (
                y_train_full
                if y_train_full.nunique() > 1 and y_train_full.value_counts().min() >= 2
                else None
            )

            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_train
            )

            stratify_test = (
                y_test_full
                if y_test_full.nunique() > 1 and y_test_full.value_counts().min() >= 2
                else None
            )

            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_test
            )

            if y_train_split.nunique() < 2:
                continue

            scaler = StandardScaler()

            X_train_s = scaler.fit_transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            try:
                clf.fit(X_train_s, y_train_split)

                y_pred = clf.predict(X_test_s)

                accuracy_scores.append(
                    accuracy_score(y_test_split, y_pred)
                )

                f1_scores.append(
                    f1_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                precision_scores.append(
                    precision_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                recall_scores.append(
                    recall_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

            except Exception:
                continue

        if len(accuracy_scores) == 0:
            results.append({
                "Model": name,
                "Accuracy Mean": np.nan,
                "Accuracy Std": np.nan,
                "F1 Mean": np.nan,
                "F1 Std": np.nan,
                "Precision Mean": np.nan,
                "Precision Std": np.nan,
                "Recall Mean": np.nan,
                "Recall Std": np.nan,
                "Accuracy ± SD": "N/A",
                "F1 ± SD": "N/A",
                "Precision ± SD": "N/A",
                "Recall ± SD": "N/A",
                "Accuracy (Mean±Std)": "N/A",
                "F1 (Mean±Std)": "N/A",
                "Precision (Mean±Std)": "N/A",
                "Recall (Mean±Std)": "N/A"
            })
            continue

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1) if len(accuracy_scores) > 1 else 0

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1) if len(f1_scores) > 1 else 0

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1) if len(precision_scores) > 1 else 0

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1) if len(recall_scores) > 1 else 0

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy ± SD": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 ± SD": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision ± SD": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall ± SD": f"{rec_mean:.4f} ± {rec_std:.4f}",

            "Accuracy (Mean±Std)": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 (Mean±Std)": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision (Mean±Std)": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall (Mean±Std)": f"{rec_mean:.4f} ± {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False,
        na_position="last"
    )

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        scaler = StandardScaler()

        X_train_real = scaler.fit_transform(X_train_real)
        X_test_real = scaler.transform(X_test_real)

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)


--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: y
Target classes:
y
0    8794
1    1206
Name: count, dtype: int64
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
1,SVM-RBF,0.8889 ± 0.0036,0.8591 ± 0.0053,0.8668 ± 0.0082,0.8889 ± 0.0036
0,LogReg,0.8885 ± 0.0028,0.8570 ± 0.0045,0.8666 ± 0.0070,0.8885 ± 0.0028
7,GradientBoost,0.8884 ± 0.0036,0.8594 ± 0.0045,0.8657 ± 0.0082,0.8884 ± 0.0036
8,AdaBoost,0.8883 ± 0.0041,0.8584 ± 0.0056,0.8656 ± 0.0097,0.8883 ± 0.0041
9,MLP,0.8848 ± 0.0055,0.8586 ± 0.0069,0.8587 ± 0.0107,0.8848 ± 0.0055
5,RandomForest,0.8831 ± 0.0058,0.8575 ± 0.0066,0.8557 ± 0.0106,0.8831 ± 0.0058
2,KNN,0.8797 ± 0.0028,0.8544 ± 0.0038,0.8496 ± 0.0051,0.8797 ± 0.0028
6,ExtraTrees,0.8702 ± 0.0046,0.8487 ± 0.0047,0.8392 ± 0.0065,0.8702 ± 0.0046
3,NaiveBayes,0.8510 ± 0.0078,0.8419 ± 0.0068,0.8348 ± 0.0068,0.8510 ± 0.0078
4,DecisionTree,0.8092 ± 0.0098,0.8142 ± 0.0068,0.8197 ± 0.0046,0.8092 ± 0.0098


In [9]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "ForestDiffusion",
    "TabDDPM"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label_col="y",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy ± SD",
            "F1 ± SD",
            "Precision ± SD",
            "Recall ± SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    if label_col not in synthetic_train_df.columns:
        print(f"{label_col} not found in {synth_name}. Skipping.")
        continue

    synthetic_train_df[label_col] = pd.to_numeric(
        synthetic_train_df[label_col],
        errors="coerce"
    )

    synthetic_train_df[label_col] = (
        synthetic_train_df[label_col]
        .fillna(real_data[label_col].mode()[0])
        .round()
        .astype(int)
    )

    synthetic_train_df = synthetic_train_df.dropna()

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label_col="y",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy ± SD",
                "F1 ± SD",
                "Precision ± SD",
                "Recall ± SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy ± SD_TRTR",
                "Accuracy ± SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby(
        "Synthetic_Model",
        as_index=False
    )[
        [
            "Accuracy_Drop",
            "F1_Drop",
            "Precision_Drop",
            "Recall_Drop"
        ]
    ]
    .mean()
    .sort_values(
        "Accuracy_Drop",
        ascending=True
    )
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
1,SVM-RBF,0.8889 ± 0.0036,0.8591 ± 0.0053,0.8668 ± 0.0082,0.8889 ± 0.0036
0,LogReg,0.8885 ± 0.0028,0.8570 ± 0.0045,0.8666 ± 0.0070,0.8885 ± 0.0028
7,GradientBoost,0.8884 ± 0.0036,0.8594 ± 0.0045,0.8657 ± 0.0082,0.8884 ± 0.0036
8,AdaBoost,0.8883 ± 0.0041,0.8584 ± 0.0056,0.8656 ± 0.0097,0.8883 ± 0.0041
9,MLP,0.8848 ± 0.0055,0.8586 ± 0.0069,0.8587 ± 0.0107,0.8848 ± 0.0055
5,RandomForest,0.8831 ± 0.0058,0.8575 ± 0.0066,0.8557 ± 0.0106,0.8831 ± 0.0058
2,KNN,0.8797 ± 0.0028,0.8544 ± 0.0038,0.8496 ± 0.0051,0.8797 ± 0.0028
6,ExtraTrees,0.8702 ± 0.0046,0.8487 ± 0.0047,0.8392 ± 0.0065,0.8702 ± 0.0046
3,NaiveBayes,0.8510 ± 0.0078,0.8419 ± 0.0068,0.8348 ± 0.0068,0.8510 ± 0.0078
4,DecisionTree,0.8092 ± 0.0098,0.8142 ± 0.0068,0.8197 ± 0.0046,0.8092 ± 0.0098


CTGAN not found in synthetic_datasets. Skipping.
CopulaGAN not found in synthetic_datasets. Skipping.
TVAE not found in synthetic_datasets. Skipping.
GaussianCopula not found in synthetic_datasets. Skipping.
ForestDiffusion - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
6,ExtraTrees,0.8929 ± 0.0060,0.8694 ± 0.0082,0.8737 ± 0.0109,0.8929 ± 0.0060
1,SVM-RBF,0.8890 ± 0.0029,0.8586 ± 0.0050,0.8672 ± 0.0067,0.8890 ± 0.0029
7,GradientBoost,0.8887 ± 0.0052,0.8652 ± 0.0070,0.8661 ± 0.0095,0.8887 ± 0.0052
8,AdaBoost,0.8886 ± 0.0033,0.8589 ± 0.0053,0.8660 ± 0.0075,0.8886 ± 0.0033
0,LogReg,0.8880 ± 0.0023,0.8562 ± 0.0042,0.8650 ± 0.0053,0.8880 ± 0.0023
5,RandomForest,0.8869 ± 0.0067,0.8664 ± 0.0079,0.8642 ± 0.0111,0.8869 ± 0.0067
2,KNN,0.8830 ± 0.0073,0.8627 ± 0.0088,0.8584 ± 0.0114,0.8830 ± 0.0073
9,MLP,0.8820 ± 0.0055,0.8644 ± 0.0066,0.8591 ± 0.0084,0.8820 ± 0.0055
3,NaiveBayes,0.8449 ± 0.0071,0.8390 ± 0.0066,0.8339 ± 0.0070,0.8449 ± 0.0071
4,DecisionTree,0.8111 ± 0.0181,0.8186 ± 0.0134,0.8273 ± 0.0085,0.8111 ± 0.0181


ForestDiffusion - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogReg,0.8795 ± 0.0000,0.8231 ± 0.0000,0.7735 ± 0.0000,0.8795 ± 0.0000
1,SVM-RBF,0.8795 ± 0.0000,0.8231 ± 0.0000,0.7735 ± 0.0000,0.8795 ± 0.0000
8,AdaBoost,0.8795 ± 0.0000,0.8231 ± 0.0000,0.7735 ± 0.0000,0.8795 ± 0.0000
5,RandomForest,0.8794 ± 0.0002,0.8231 ± 0.0001,0.7735 ± 0.0000,0.8794 ± 0.0002
6,ExtraTrees,0.8764 ± 0.0019,0.8228 ± 0.0012,0.7889 ± 0.0127,0.8764 ± 0.0019
7,GradientBoost,0.8759 ± 0.0053,0.8226 ± 0.0024,0.7858 ± 0.0239,0.8759 ± 0.0053
2,KNN,0.8726 ± 0.0054,0.8212 ± 0.0024,0.7873 ± 0.0125,0.8726 ± 0.0054
9,MLP,0.8526 ± 0.0554,0.8121 ± 0.0240,0.7908 ± 0.0379,0.8526 ± 0.0554
3,NaiveBayes,0.8526 ± 0.0509,0.8106 ± 0.0253,0.7761 ± 0.0055,0.8526 ± 0.0509
4,DecisionTree,0.7089 ± 0.0809,0.7403 ± 0.0535,0.7872 ± 0.0112,0.7089 ± 0.0809


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,TabDDPM,SVM-RBF,0.00940,0.036000,0.093292,0.00940,0.8889 ± 0.0036,0.8795 ± 0.0000
1,TabDDPM,LogReg,0.00900,0.033917,0.093042,0.00900,0.8885 ± 0.0028,0.8795 ± 0.0000
2,TabDDPM,GradientBoost,0.01260,0.036824,0.079913,0.01260,0.8884 ± 0.0036,0.8759 ± 0.0053
3,TabDDPM,AdaBoost,0.00885,0.035286,0.092063,0.00885,0.8883 ± 0.0041,0.8795 ± 0.0000
4,TabDDPM,MLP,0.03220,0.046498,0.067873,0.03220,0.8848 ± 0.0055,0.8526 ± 0.0554
5,TabDDPM,RandomForest,0.00375,0.034416,0.082142,0.00375,0.8831 ± 0.0058,0.8794 ± 0.0002
6,TabDDPM,KNN,0.00700,0.033182,0.062326,0.00700,0.8797 ± 0.0028,0.8726 ± 0.0054
7,TabDDPM,ExtraTrees,-0.00625,0.025883,0.050351,-0.00625,0.8702 ± 0.0046,0.8764 ± 0.0019
8,TabDDPM,NaiveBayes,-0.00155,0.031284,0.058739,-0.00155,0.8510 ± 0.0078,0.8526 ± 0.0509
9,TabDDPM,DecisionTree,0.10030,0.073874,0.032499,0.10030,0.8092 ± 0.0098,0.7089 ± 0.0809


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,-0.002295,-0.005025,-0.005855,-0.002295
1,TabDDPM,0.017530,0.038716,0.071224,0.017530


In [10]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
